# Visualization of bilipschitz activations

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from dlroms import dv

from activations import LeakyReLU, HypAct

### Compute activation outputs

In [ ]:
# Generate ranges
x_lin = dv.tensor(np.linspace(-3,3,1001))
alphas_leaky = np.linspace(1/3,3,30)
betas_leaky = np.linspace(1/3,3,30)
thetas_hypact = np.linspace(0.1, np.pi / 4 - 0.1, 30)

# LeakyReLU_{\alpha,1}
ys_leaky_alpha = list()
for alpha in alphas_leaky:
    ys_leaky_alpha.append(LeakyReLU(alpha).act(x_lin).detach().cpu().numpy())

# LeakyReLU_{1,\beta}
ys_leaky_beta = list()
for beta in betas_leaky:
    ys_leaky_beta.append(LeakyReLU(1,beta).act(x_lin).detach().cpu().numpy())

# Hypact_{\theta}
ys_hypact = list()
for theta in thetas_hypact:
    ys_hypact.append(HypAct(theta).act(x_lin).detach().cpu().numpy())

# Postprocess
x_lin = x_lin.detach().cpu().numpy()
ys_leaky_alpha = np.array(ys_leaky_alpha)
ys_leaky_beta = np.array(ys_leaky_beta)
ys_hypact = np.array(ys_hypact)

### Visualize activation effect

In [ ]:
# Plots activation to the corresponding axis
def plot_activation(
    ax,
    cmap,
    act_parameter_values : np.array,
    y_values : np.array,
    act_param_str : str,
    act_str : str,
    labelpad : float = -40
):
    colors = cmap(np.linspace(0, 1, y_values.shape[0]))
    for i, color in enumerate(colors):
        ax.plot(x_lin, y_values[i], color = color)
    ax.grid(color = 'gray', linestyle = ':', linewidth = 0.3)
    ax.set_xlim([-3,3])
    ax.set_ylim([-3,3])
    ax.hlines(0, xmin = -3, xmax = 3, color = 'k', linewidth = 0.5)
    ax.vlines(0, ymin = -3, ymax = 3, color = 'k', linewidth = 0.5)
    norm = mpl.colors.Normalize(
        vmin = act_parameter_values.min(), vmax = act_parameter_values.max()
    )
    act_param_mappable = mpl.cm.ScalarMappable(norm = norm, cmap = cmap)
    cax = inset_axes(ax, width = "3%", height = "30%", loc = 4, borderpad = 3)
    cbar = fig.colorbar(act_param_mappable, ax = ax, cax = cax)
    cbar.set_label(
        act_param_str, rotation = 90, labelpad = labelpad, fontsize = 14
    )
    ax.set_xlabel(act_str, fontsize = 15)
    cax.tick_params(axis = 'both', which = 'major', labelsize = 13)
    ax.tick_params(
        axis = 'both', which = 'both', labelsize = 14, colors = 'gray'
    )


# Create figure
fig, ax = plt.subplots(1, 3, figsize = (18,5))

# LeakyReLU_{\alpha,1}
plot_activation(
    ax = ax[0],
    cmap = mpl.colormaps['Greens'],
    act_parameter_values = alphas_leaky,
    y_values = ys_leaky_alpha,
    act_param_str = '$\\alpha$',
    act_str = '$\\text{LeakyReLU}_{\\alpha,1}$'
)


# LeakyReLU_{1,\beta}
plot_activation(
    ax = ax[1],
    cmap = mpl.colormaps['Reds'],
    act_parameter_values = betas_leaky,
    y_values = ys_leaky_beta,
    act_param_str = '$\\beta$',
    act_str = '$\\text{LeakyReLU}_{1,\\beta}$'
)


# Hypact_{\theta}
plot_activation(
    ax = ax[2],
    cmap = mpl.colormaps['Purples'],
    act_parameter_values = thetas_hypact,
    y_values = ys_hypact,
    act_param_str = '$\\theta$',
    act_str = '$\\text{HypAct}_{\\theta}$',
    labelpad = -52
)

# Save figure
savepath = os.path.join('..', 'results', 'figures')
if not os.path.exists(savepath):
    os.makedirs(savepath)
plt.savefig(os.path.join(savepath, 'activations.png'))